# 01 Baseline Colab

本 notebook 用于在 Colab 中跑通 Qwen3-ASR-1.7B baseline smoke 评估。当前目标是验证最小闭环：Google Drive 路径、clean/noise 音频、manifest、Qwen3-ASR 推理、prediction JSONL、WER/CER 指标。

注意：2 条 smoke 样本只能证明流程可运行，不能证明模型鲁棒性已经达标。后续需要扩展到 clean、noise、reverb、far_field、dropout 等场景。

In [ ]:
# 挂载 Google Drive。
# 所有输入音频、manifest、输出 prediction 和指标都会读写到 Drive，避免 Colab runtime 重启后丢失。
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# 安装最小依赖。
# qwen-asr 是 Qwen3-ASR 官方推理包；only-if-needed 可以尽量避免升级 Colab 预装的 numpy、requests、click。
# 不要安装未固定版本的 pandas；Colab 当前依赖 pandas==2.2.2，pandas 3.x 会和 google-colab/db-dtypes/gradio 冲突。
# pyyaml 用于读取 configs/baseline/qwen3_asr_baseline.yaml。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr soundfile huggingface_hub pyyaml
%pip -q install pandas==2.2.2


In [ ]:
# 读取项目路径和 baseline 配置。
# 只要你的 Drive 目录名不一样，通常只需要改 PROJECT_DIR 这一行。
from pathlib import Path
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/baseline/qwen3_asr_baseline.yaml'

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)


def project_path(value: str) -> Path:
    """把配置中的相对路径解析到 PROJECT_DIR 下，绝对路径保持不变。"""
    path = Path(value)
    return path if path.is_absolute() else PROJECT_DIR / path

# smoke 音频目录：放 clean_0001.wav 和 noise_0001.wav。
AUDIO_DIR = PROJECT_DIR / 'audio/smoke'

# 输入、输出路径来自 baseline 配置，保证 notebook 和脚本命令口径一致。
MANIFEST = project_path(config['input']['manifest'])
AUDIO_ROOT = project_path(config['input']['audio_root'])
PREDICTIONS = project_path(config['output']['predictions_jsonl'])
SCORED = project_path(config['output']['scored_jsonl'])
METRICS = project_path(config['output']['metrics_json'])
SCENARIO_CSV = project_path(config['output']['metrics_by_scenario_csv'])
OUTPUT_DIR = PREDICTIONS.parent

# 模型和推理参数也从配置读取；后续 baseline 对比时优先改 YAML。
MODEL_ID = config['model']['id']
DTYPE = config['model'].get('dtype', 'float16')
DEVICE_MAP = config['model'].get('device_map', 'cuda:0')
MAX_INFERENCE_BATCH_SIZE = int(config['model'].get('max_inference_batch_size', 1))
INFERENCE = config.get('inference', {})
LANGUAGE = INFERENCE.get('language', 'English')
MAX_NEW_TOKENS = int(INFERENCE.get('max_new_tokens', 128))
LIMIT = int(config['runtime'].get('limit', 0))

# 创建必要目录。checkpoints/logs 是后续 LoRA 训练会复用的目录，这里先准备好。
for subdir in [
    AUDIO_DIR,
    MANIFEST.parent,
    OUTPUT_DIR,
    PROJECT_DIR / 'checkpoints',
    PROJECT_DIR / 'logs',
]:
    subdir.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG_PATH =', CONFIG_PATH)
print('MANIFEST =', MANIFEST)
print('AUDIO_ROOT =', AUDIO_ROOT)
print('PREDICTIONS =', PREDICTIONS)
print('MODEL_ID =', MODEL_ID)
print('LANGUAGE =', LANGUAGE)


In [ ]:
# 检查 Drive 中是否已经上传 smoke 音频。
# 当前 notebook 默认使用两条音频：一条 clean，一条 noise/degraded。
# 如果这里 assert 失败，先把音频上传到 PROJECT_DIR/audio/smoke/。
clean_audio = AUDIO_DIR / 'clean_0001.wav'
noise_audio = AUDIO_DIR / 'noise_0001.wav'

print('当前 smoke 音频目录文件：')
for path in sorted(AUDIO_DIR.glob('*')):
    print(' -', path)

assert clean_audio.exists(), f'找不到 clean 音频: {clean_audio}'
assert noise_audio.exists(), f'找不到 noise 音频: {noise_audio}'

In [ ]:
# 生成 baseline smoke manifest。
# answer 必须和音频真实内容一致，否则 WER/CER 没有意义。
# 这里使用绝对音频路径，Colab 中最直观；后续扩大数据集时也可以改成相对路径 + audio_root。
import json

clean_text = 'Please call me when the meeting starts.'

samples = [
    {
        'audio': str(clean_audio),
        'answer': clean_text,
        'language': 'en',
        'scenario': 'clean',
        'source': 'local_smoke',
        'is_degraded': False,
    },
    {
        'audio': str(noise_audio),
        'answer': clean_text,
        'language': 'en',
        'scenario': 'noise',
        'source': 'local_smoke',
        'is_degraded': True,
    },
]

with MANIFEST.open('w', encoding='utf-8') as f:
    for item in samples:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print('已写入 manifest:', MANIFEST)



In [ ]:
# 预览并校验 manifest。
# 这个单元用于在真正加载 Qwen3-ASR 模型前提前发现路径错误、字段缺失或音频未上传问题。
rows = []
with MANIFEST.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

assert rows, 'manifest 为空'
for row in rows:
    assert 'audio' in row, f'缺少 audio 字段: {row}'
    assert 'answer' in row, f'缺少 answer 字段: {row}'
    assert 'scenario' in row, f'缺少 scenario 字段: {row}'
    assert Path(row['audio']).exists(), f"音频路径不存在: {row['audio']}"
    print(row)

In [ ]:
# 登录 Hugging Face。
# 如果 Hugging Face 下载限流或需要认证，可以在这里登录后重试。
# 如果你已经在 Colab Secrets 中配置 HF_TOKEN，也可以跳过手动输入。
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 运行 Qwen3-ASR-1.7B baseline 推理。
# 这里使用 subprocess 列表参数，而不是 shell 字符串，避免 {MANIFEST} 这类变量没有展开的问题。
# LIMIT=2 表示先只跑 clean/noise 两条 smoke 样本。
import subprocess
import sys

cmd = [
    sys.executable,
    'inference/qwen3_asr_base_infer.py',
    '--manifest', str(MANIFEST),
    '--audio-root', str(AUDIO_ROOT),
    '--output-jsonl', str(PREDICTIONS),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--max-inference-batch-size', str(MAX_INFERENCE_BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--language', LANGUAGE,
]
if LIMIT > 0:
    cmd.extend(['--limit', str(LIMIT)])

print('运行命令:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)
print('prediction JSONL =', PREDICTIONS)


In [ ]:
# 预览模型输出。
# prediction 是模型转写，error 为空表示这条样本推理过程没有异常。
with PREDICTIONS.open('r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        print({
            'scenario': row.get('scenario'),
            'answer': row.get('answer'),
            'prediction': row.get('prediction'),
            'error': row.get('error'),
        })

In [ ]:
# 计算 WER/CER 指标。
# eval_wer.py 会输出三类文件：逐条 scored JSONL、整体 metrics JSON、按场景聚合 CSV。
eval_cmd = [
    sys.executable,
    'evaluation/eval_wer.py',
    '--predictions-jsonl', str(PREDICTIONS),
    '--scored-jsonl', str(SCORED),
    '--metrics-json', str(METRICS),
    '--metrics-by-scenario-csv', str(SCENARIO_CSV),
]

print('运行命令:')
print(' '.join(eval_cmd))
subprocess.run(eval_cmd, cwd=str(PROJECT_DIR), check=True)

print('metrics JSON =', METRICS)
print(METRICS.read_text(encoding='utf-8'))

In [ ]:
# 查看按场景聚合的得分表。
# clean/noise 的 error_rate 都是 0.0 时，说明两条 smoke 样本完全匹配 reference。
# 这只是流程验收，后续还要扩大到更多样本和退化场景。
import pandas as pd

pd.read_csv(SCENARIO_CSV)